**Note:**  
This notebook expects the raw dataset to be located at `../data/fraud_data_raw.csv`.  
Please place the dataset in the `data1/` folder before running.

#Preprocessing for Fraud Detection

In this notebook, we prepare the synthetic bank transaction dataset for modeling by:
- Dropping irrelevant or non-predictive columns
- Encoding categorical variables
- Ensuring all features are numeric and model-ready
- Saving a cleaned dataset for the modeling phase

In [1]:
#Import & Settings

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)


In [2]:
#Load Raw Dataset - starting from the original transaction dataset and building from there.
df = pd.read_csv("../data/fraud_data_raw.csv")

df.head()




,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 11 columns):
 #   Column          Dtype  
---  ------          -----  
 0   step            int64  
 1   type            str    
 2   amount          float64
 3   nameOrig        str    
 4   oldbalanceOrg   float64
 5   newbalanceOrig  float64
 6   nameDest        str    
 7   oldbalanceDest  float64
 8   newbalanceDest  float64
 9   isFraud         int64  
 10  isFlaggedFraud  int64  
dtypes: float64(5), int64(3), str(3)
memory usage: 534.0 MB


In [4]:
#Additional Data Checks: Confirming structure, data types, missing values

df.describe(include="all")


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
count,6.362620e+06,6362620,6.362620e+06,6362620,6.362620e+06,6.362620e+06,6362620,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06
unique,NaN,5,NaN,6353307,NaN,NaN,2722362,NaN,NaN,NaN,NaN
top,NaN,CASH_OUT,NaN,C2098525306,NaN,NaN,C1286084959,NaN,NaN,NaN,NaN
freq,NaN,2237500,NaN,3,NaN,NaN,113,NaN,NaN,NaN,NaN
mean,2.433972e+02,NaN,1.798619e+05,NaN,8.338831e+05,8.551137e+05,NaN,1.100702e+06,1.224996e+06,1.290820e-03,2.514687e-06
std,1.423320e+02,NaN,6.038582e+05,NaN,2.888243e+06,2.924049e+06,NaN,3.399180e+06,3.674129e+06,3.590480e-02,1.585775e-03
min,1.000000e+00,NaN,0.000000e+00,NaN,0.000000e+00,0.000000e+00,NaN,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,1.560000e+02,NaN,1.338957e+04,NaN,0.000000e+00,0.000000e+00,NaN,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
50%,2.390000e+02,NaN,7.487194e+04,NaN,1.420800e+04,0.000000e+00,NaN,1.327057e+05,2.146614e+05,0.000000e+00,0.000000e+00
75%,3.350000e+02,NaN,2.087215e+05,NaN,1.073152e+05,1.442584e+05,NaN,9.430367e+05,1.111909e+06,0.000000e+00,0.000000e+00


In [5]:
#Dropping non-predictive ID and "rule-based" comments
#The following columns are not useful for modeling:
### "nameOrig" and "nameDest" are highly-unique character IDs.
## "isFlaggedFraud" is a synthetic rule that rarely triggers and does not help detect fraud.

cols_to_drop = ["nameOrig", "nameDest", "isFlaggedFraud"]
df = df.drop(columns=cols_to_drop)

df.head()
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 8 columns):
 #   Column          Dtype  
---  ------          -----  
 0   step            int64  
 1   type            str    
 2   amount          float64
 3   oldbalanceOrg   float64
 4   newbalanceOrig  float64
 5   oldbalanceDest  float64
 6   newbalanceDest  float64
 7   isFraud         int64  
dtypes: float64(5), int64(2), str(1)
memory usage: 388.3 MB


In [6]:
#Encode trnasaction type
#As shown in the EDA, The "type" column is categorical with 5 transaction types. We can convert it to "dummy variables" for modeling.

df["type"].value_counts()


type
CASH_OUT    2237500
PAYMENT     2151495
CASH_IN     1399284
TRANSFER     532909
DEBIT         41432
Name: count, dtype: int64

In [7]:
df = pd.get_dummies(df, columns=["type"], drop_first=True)

df.head()
df.info()

#For a categorical variable with N categories, we should only include N - 1 dummy variables in our model , dropping one baseline category. This ensure features avoids becoming multicollinear, so that one variable can not be flawlessly predicted by the others. The omitted category becomes the reference point, and its value is absorbed into the model's baseline intercept.


<class 'pandas.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 11 columns):
 #   Column          Dtype  
---  ------          -----  
 0   step            int64  
 1   amount          float64
 2   oldbalanceOrg   float64
 3   newbalanceOrig  float64
 4   oldbalanceDest  float64
 5   newbalanceDest  float64
 6   isFraud         int64  
 7   type_CASH_OUT   bool   
 8   type_DEBIT      bool   
 9   type_PAYMENT    bool   
 10  type_TRANSFER   bool   
dtypes: bool(4), float64(5), int64(2)
memory usage: 364.1 MB


## Feature engineering: balance differences

From EDA, we saw that:
- Fraud often drains the origin account.
- Old and new balances are highly correlated.

We create features that capture balance changes:
- `balanceDiffOrg = oldbalanceOrg - newbalanceOrig`
- `balanceDiffDest = newbalanceDest - oldbalanceDest`

In [ ]:
df["balanceDiffOrg"] = df["oldbalanceOrg"] - df["newbalanceOrig"]
df["balanceDiffDest"] = df["newbalanceDest"] - df["oldbalanceDest"]


df[["oldbalanceDest", "newbalanceDest", "balanceDiffDest"]].head()


,oldbalanceDest,newbalanceDest,balanceDiffDest
0,0.0,0.0,0.0
1,0.0,0.0,0.0
2,0.0,0.0,0.0
3,21182.0,0.0,-21182.0
4,0.0,0.0,0.0


In [9]:
df[["oldbalanceOrg", "newbalanceOrig", "balanceDiffOrg"]].head()

,oldbalanceOrg,newbalanceOrig,balanceDiffOrg
0,170136.0,160296.36,9839.64
1,21249.0,19384.72,1864.28
2,181.0,0.00,181.00
3,181.0,0.00,181.00
4,41554.0,29885.86,11668.14


In [ ]:
#Final dataset check
#We confirm: All features are numeric, No object dtypes remain, Target `isFraud` is present and correctly typed.Boolean characters are expected and acceptable for modeling_note: Machine learning models will treat them like "ints"

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 13 columns):
 #   Column           Dtype  
---  ------           -----  
 0   step             int64  
 1   amount           float64
 2   oldbalanceOrg    float64
 3   newbalanceOrig   float64
 4   oldbalanceDest   float64
 5   newbalanceDest   float64
 6   isFraud          int64  
 7   type_CASH_OUT    bool   
 8   type_DEBIT       bool   
 9   type_PAYMENT     bool   
 10  type_TRANSFER    bool   
 11  balanceDiffOrg   float64
 12  balanceDiffDest  float64
dtypes: bool(4), float64(7), int64(2)
memory usage: 461.2 MB


In [12]:
df.to_csv("../data/fraud_data_cleaned.csv", index=False)
print("Saved cleaned dataset to ../data/fraud_data_cleaned.csv")


Saved cleaned dataset to ../data/fraud_data_cleaned.csv
